In [1]:
from GradientGang.Pipeline.FinalPipeline import FinalPipeline

## Initialize FinalPipeline

In [2]:
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 128,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
    'use_kfold': True,
    'n_folds': 4,
}

In [3]:
# Initialize FinalPipeline with parameters
import os
import dotenv
env_path = '../src/GradientGang/Pipeline/Optimizer/.env'
dotenv.load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")

In [4]:
params = {
    "database_url": database_url,
    "data_params": data_params,
    'project_name': 'pirate_pain_classification',
    'study_name': '_theBeastComputerGP_AugmentedTest',
    'submission_path': '../Submissions',
}

pipeline = FinalPipeline(params)

Storage: postgresql://postgres...
✓ Database configuration loaded
✓ Database initialized successfully!


[I 2025-11-15 11:26:09,663] Using an existing study with name 'pirate_pain_classification_theBeastComputerGP_AugmentedTest' instead of creating a new one.


✓ Study created/loaded successfully!
Study name: pirate_pain_classification_theBeastComputerGP_AugmentedTest
Sampler: GPSampler
Pruner: MedianPruner
Storage: Database
Total trials: 5
✓ Study initialized successfully!
✓ FinalPipeline initialized successfully!


## RUN FinalPipeline

## Debug: Test augmentation disabled

Testing if the error persists with augmentation disabled. If it does, the issue is in the model architecture, not augmentation.

## Fix Applied - Complete ✅
**Issue**: Tensor size mismatch (5440 vs 340) during training
- **Root cause**: When using WindowedModelWrapper with autoencoder, the base model's decoder was configured to reconstruct full 160-length sequences, but WindowedModelWrapper passes it windowed sequences (e.g., length 10)
- **Solution**: Adjusted datasetInfo to reflect what the base model actually receives when windowing is enabled
  - `datasetInfo["timeSeriesShape"]` now shows `(34, window_size)` instead of `(34, 160)` when `use_windowing=True`
  - Applied fix in 3 locations:
    1. `objective_kfold()` - During Optuna optimization (line ~1181)
    2. `load_ensemble_model()` - When loading ensemble (line ~1935)
    3. `create_submission()` - When creating submissions (line ~2210)

**Key architectural principle**: 
- DataLoader **always** provides full sequences (160 timesteps) with `use_windowing=False`
- `WindowedModelWrapper` handles windowing on the **model side only**
- Base model architecture must be built for windowed dimensions when wrapped

In [ ]:
pipeline.optuna_optimize(n_trials=100000000000)  # Start the optimization process

AttributeError: 'FinalPipeline' object has no attribute 'optimize'

## Track FinalPipeline

In [ ]:
pipeline.study_summary()  # Display study summary

## Submit FinalPipeline

In [ ]:
pipeline.create_submission()  # Generate submission file